# Imports

In [2]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

In [ ]:
# nltk.download()
# nltk.download('punkt_tab')

# Load Data

Every row is a review of a movie from a IMDB user.

In [3]:
df = pd.read_csv("data/dataset.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['sentiment'].value_counts()

# Data Cleaning

## Target Variable

In [ ]:
def replace(x):
    if x == 'positive':
        return 1
    if x == 'negative':
        return 0

In [ ]:
df['sentiment'] = df['sentiment'].map(replace)

In [ ]:
df['sentiment']

In [ ]:
df['review'][0]

### Clean html tags

In [ ]:
def clean_html(texto):
    pat = re.compile(r'<.*?>')
    return re.sub(pattern = pat, repl = '', string = texto)

In [ ]:
df['review'] = df['review'].apply(clean_html)

In [ ]:
df['review'][0]

### Clean special chars

In [ ]:
def clean_special_chars(txt):
    rem = ''
    for e in txt:
        if e.isalnum():
            rem = rem + e
        else:
            rem = rem + ' '
    return rem

In [ ]:
df['review'] = df['review'].apply(clean_special_chars)

In [ ]:
df['review'][0]

### Lower

In [ ]:
def force_lower(texto):
    return texto.lower()

In [ ]:
df['review'] = df['review'].apply(force_lower)

In [ ]:
df['review'][0]

### Remove stopwords

In [ ]:
stopwords.words('english')[:10]

In [ ]:
def remove_stopwords(texto):
    stop_words = stopwords.words('english')
    words = word_tokenize(texto)
    return [w for w in words if w not in stop_words]

In [ ]:
df['review'] = df['review'].apply(remove_stopwords)

In [ ]:
df['review'][0]

### Remove affixes (prefixes and suffixes)

Stemming is a Natural Language Processing (NLP) technique used to reduce words to their root or base form by removing common prefixes and suffixes. This helps normalize words for analysis and can improve the performance of NLP tasks such as text search, text classification, and document clustering.

In [ ]:
SnowballStemmer('portuguese').stem('assoviando')

In [ ]:
def stem(texto):
    obj = SnowballStemmer('english')
    return ' '.join([obj.stem(w) for w in texto])

In [ ]:
txt = 'The cats are sleeping and the dogs are eating'.split()
txt

In [ ]:
# Test the function
stem(txt)

In [ ]:
df['review'] = df['review'].apply(stem)

In [ ]:
df['review'][0]

# Pre-processing

In [ ]:
df.sample(10)

In [ ]:
X = df['review']

In [ ]:
y = df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [ ]:
X_train.iloc[0]

#  Model Selection

In [ ]:
vectorizer = CountVectorizer(max_features=1000)

In [ ]:
vectorizer.fit(X_train)

In [ ]:
vectorizer.get_feature_names_out()

In [ ]:
X_train_final = vectorizer.transform(X_train).toarray()

In [ ]:
X_train_final[0,:15]

In [ ]:
X_test_final = vectorizer.transform(X_test).toarray()

## Gaussian Naive Bayes

In [ ]:
gaussian_nb = GaussianNB()

In [ ]:
gaussian_nb.fit(X_train_final, y_train)

## Multinomial Naive Bayes

In [ ]:
multinomial_nb = MultinomialNB()

In [ ]:
multinomial_nb.fit(X_train_final, y_train)

## Bernoulli Naive Bayes

In [ ]:
bernoulli_nb = BernoulliNB()

In [ ]:
bernoulli_nb.fit(X_train_final, y_train)

# Model Evaluation

In [ ]:
y_pred_gaussian = gaussian_nb.predict(X_test_final)

In [ ]:
y_pred_mult = multinomial_nb.predict(X_test_final)

In [ ]:
y_pred_bernoulli = bernoulli_nb.predict(X_test_final)

In [ ]:
models = ['Gaussian', 'Multinomial', 'Bernoulli']
predctions = [y_pred_gaussian, y_pred_mult, y_pred_bernoulli]

## Accuracy

In [ ]:
for m, predict in zip(models, predctions):
    print(f'Model {m} -> Accuracy: {accuracy_score(y_test, predict)}')

## AUC

In [ ]:
prob_pred_gaussian = gaussian_nb.predict_proba(X_test_final)
prob_pred_mult = multinomial_nb.predict_proba(X_test_final)
prob_pred_bernoulli = bernoulli_nb.predict_proba(X_test_final)

In [ ]:
predict_probab = [prob_pred_gaussian, prob_pred_mult, prob_pred_bernoulli]

In [ ]:
for m, prob in zip(models, predict_probab):
    print(f' Modelo {m} --> AUC: {roc_auc_score(y_test, prob[:,1]):.4f}')

In [ ]:
best_model = bernoulli_nb

# Pipeline

In [ ]:
def transform(txt):
    a = clean_html(txt)
    b = clean_special_chars(a)
    c = force_lower(b)
    d = remove_stopwords(c)
    e = stem(d)
    f = vectorizer.transform(np.array([e])).toarray()
    return f

# Inference

## Text 1

In [ ]:
txt_1 = """This is probably the fastest-paced and most action-packed of the German Edgar Wallace ""krimi"" 
series, a cross between the Dr. Mabuse films of yore and 60's pop thrillers like Batman and the Man 
from UNCLE. It reintroduces the outrageous villain from an earlier film who dons a stylish monk's habit and 
breaks the necks of victims with the curl of a deadly whip. Set at a posh girls' school filled with lecherous 
middle-aged professors, and with the cops fondling their hot-to-trot secretaries at every opportunity, it 
certainly is a throwback to those wonderfully politically-incorrect times. There's a definite link to a later 
Wallace-based film, the excellent giallo ""Whatever Happened to Solange?"", which also concerns female students 
being corrupted by (and corrupting?) their elders. Quite appropriate to the monk theme, the master-mind villain 
uses booby-trapped bibles here to deal some of the death blows, and also maintains a reptile-replete dungeon 
to amuse his captive audiences. <br /><br />Alfred Vohrer was always the most playful and visually flamboyant 
of the series directors, and here the lurid colour cinematography is the real star of the show. The Monk appears 
in a raving scarlet cowl and robe, tastefully setting off the lustrous white whip, while appearing against 
purplish-night backgrounds. There's also a voyeur-friendly turquoise swimming pool which looks great both 
as a glowing milieu for the nubile students and as a shadowy backdrop for one of the murder scenes. 
The trademark ""kicker"" of hiding the ""Ende"" card somewhere in the set of the last scene is also quite 
memorable here. And there's a fine brassy and twangy score for retro-music fans.<br /><br />Fans of the series 
will definitely miss the flippant Eddie Arent character in these later films. Instead, the chief inspector 
Sir John takes on the role of buffoon, convinced that he has mastered criminal psychology after taking a few 
night courses. Unfortunately, Klaus Kinski had also gone on to bigger and better things. The krimis had 
lost some of their offbeat subversive charm by this point, and now worked on a much more blatant pop-culture 
level, which will make this one quite accessible to uninitiated viewers."""

In [ ]:
txt_1_t = transform(txt_1)

In [ ]:
txt_1_t.shape

In [ ]:
pred = best_model.predict(txt_1_t)

In [ ]:
if pred == 1:
    print('The text indicates a positive sentiment.')
else:
    print('The text indicates a negative sentiment.')

## Text 2

In [ ]:
txt_2 = """This movie is terrible, I liked very much, even tough is a good movie,\
    I really hated it because it is the best movie I have ever seen."""

In [ ]:
txt_2_t = transform(txt_2)

In [ ]:
pred = best_model.predict(txt_2_t)

In [ ]:
if pred == 1:
    print('The text indicates a positive sentiment.')
else:
    print('The text indicates a negative sentiment.')